# GPT-2 124M — DIMER text-generation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/gpt2-text-generation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/tutorials/gpt2_text_generation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-openai--community%2Fgpt2-ffcc4d?style=flat)](https://huggingface.co/openai-community/gpt2) [![Upstream](https://img.shields.io/badge/Upstream-openai%2Fgpt--2-181717?style=flat&logo=github&logoColor=white)](https://github.com/openai/gpt-2)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** causal text generation (continuing one English prompt) using the pinned GPT-2 124M weights, with greedy decoding by default and explicit, seeded nucleus sampling on request

**This notebook is standalone.** It carries the repository's pipeline module (`src/gpt2_text_generation_pipeline/pipeline.py` at revision `a9aad1e5be28`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `607a30d783dfa663caf39e06633721c8d4cfcd7e` (~554 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the byte-level BPE tokenizer turns the prompt into token ids (no special tokens are added), and the 12-layer decoder-only Transformer predicts one next-token distribution over the 50,257-token vocabulary at a time, feeding each chosen token back until `max_new_tokens` is reached or the end-of-text token is produced. **Two decoding modes are demonstrated and must not be confused (INF8):** greedy decoding (`do_sample=False`, the pipeline default) takes the argmax at every step and is deterministic on a fixed device and dtype — it is the mode for reproducibility checks and tends to repeat itself; nucleus sampling (`do_sample=True` with `temperature`, `top_p` and a mandatory `seed`) draws from the truncated distribution and is the mode usually preferred for actual use, reproducible only for the same seed on the same host. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. GPT-2 is a **base language model**: no chat template, no instruction following, no safety tuning, English web text of 2019 vintage. What the upstream checkpoint supplies is the model and tokenizer; what the carried pipeline module adds is manifest verification, input validation and ceilings (prompts are rejected, never truncated), a settings validator that refuses unseeded sampling, the fixed pad/EOS handling, a fixed output contract and the `validate_inputs` and `evaluation_report` stage helpers. **No quality metric exists** for a free-text continuation without a reference corpus; the pipeline ships no metric helper.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, author a synthetic prompt (or upload your own), stage and digest-verify the immutable upstream snapshot, surface the pipeline's ceilings and validate the prompt and both decoding settings into an input manifest before the model runs, generate a greedy continuation and confirm it is deterministic, generate a seeded sampled continuation with every setting echoed and confirm the seed reproduces it, read `finished_by` and the pad/EOS quirk correctly, read from the machine-readable evaluation report why no metric is reported and what corpus a perplexity number would need, and export machine-readable results plus provenance.

**This notebook does not demonstrate:** chat or instruction following (GPT-2 has neither), batching (one prompt per call), raw logits or hidden states, fine-tuning, beam search, non-English text, or any content filtering. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32). The model card's CPU smoke verified the 15-file snapshot in 0.32 s, loaded in 4.25 s, produced 32 greedy tokens from a 7-token prompt in 0.67 s and two seeded 16-token samples in 0.59 s together, so the default runs in seconds on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 548 MB `model.safetensors` are the largest downloads of the run.
- **Knowledge:** basic Python; what next-token prediction is; the difference between argmax decoding and sampling from a truncated distribution.
- **Data:** the default sample is one synthetic English prompt authored in code, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one UTF-8 text file whose whole content (whitespace stripped) is the prompt. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API. A base language model can continue any prompt with false, biased or offensive text — read the output before reusing it.
- **External access:** the Hugging Face Hub only, to fetch the pinned `openai-community/gpt2` snapshot (~554 MB in total) at revision `607a30d783df…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
]
NOTEBOOK_SOURCE = {
    'repository': 'gpt2-text-generation-pipeline',
    'repository_revision': 'a9aad1e5be28bb93de36156ba3fdfbbb2cd56526',
    'embedded_module': 'src/gpt2_text_generation_pipeline/pipeline.py',
    'embedded_modules': ['src/gpt2_text_generation_pipeline/pipeline.py'],
    'module_sha256': 'd1f52671b22540fe2d87e9769a65459fdf19d1638b293867966317151da0a3f5',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/gpt2_text_generation_pipeline/` @ `a9aad1e5be28`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/gpt2_text_generation_pipeline/pipeline.py`

In [ ]:
"""Causal text generation over the pinned ``openai-community/gpt2`` (124M) checkpoint.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method, ``generate``: greedy decoding by default
(deterministic), nucleus sampling only when asked for and seeded. One prompt per call.
"""

from __future__ import annotations

import hashlib
import json
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

MODEL_ID = "openai-community/gpt2"
MODEL_REVISION = "607a30d783dfa663caf39e06633721c8d4cfcd7e"
MODEL_LICENSE = "mit"
MODEL_KEY = "gpt2"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Ceilings. CONTEXT_LENGTH is n_positions / n_ctx in the pinned config.json; prompt tokens plus new tokens
# must fit in it, so a prompt is rejected (never truncated) above MAX_PROMPT_TOKENS and the combined length
# is checked before the model runs. MAX_NEW_TOKENS bounds one call's cost on CPU.
CONTEXT_LENGTH = 1024
MAX_PROMPT_TOKENS = CONTEXT_LENGTH - 1  # leaves room for at least one generated token
MAX_NEW_TOKENS = 256
DEFAULT_MAX_NEW_TOKENS = 32
MAX_TEXT_CHARS = 4_000  # pre-tokenisation guard; ~4 chars per byte-level BPE token on English text
VOCAB_SIZE = 50257  # config.json vocab_size
EOS_TOKEN_ID = 50256  # config.json eos_token_id == bos_token_id; GPT-2 has no pad token, so pad = eos
PAD_TOKEN_ID = EOS_TOKEN_ID
DECODING_DEFAULT = "greedy"  # do_sample=False -> argmax over the next-token distribution at every step


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def validate_settings(
    max_new_tokens: Any, do_sample: Any, temperature: Any, top_p: Any, seed: Any
) -> dict[str, Any]:
    """Check decoding settings before the model runs; sampling must be explicit and seeded."""
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    if not isinstance(do_sample, bool):
        raise TypeError("do_sample must be a bool")
    if isinstance(temperature, bool) or not isinstance(temperature, int | float) or not temperature > 0:
        raise ValueError("temperature must be a number > 0")
    if isinstance(top_p, bool) or not isinstance(top_p, int | float) or not 0 < top_p <= 1:
        raise ValueError("top_p must be a number in (0, 1]")
    if seed is not None and (isinstance(seed, bool) or not isinstance(seed, int) or seed < 0):
        raise TypeError("seed must be a non-negative int or None")
    if do_sample and seed is None:
        raise ValueError("seed is required when do_sample=True so that sampled output is reproducible")
    return {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "temperature": float(temperature) if do_sample else None,
        "top_p": float(top_p) if do_sample else None,
        "seed": seed if do_sample else None,
        "decoding": "nucleus-sampling" if do_sample else DECODING_DEFAULT,
    }


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one non-empty str prompt; no special tokens are added and the prompt is continued verbatim",
    "prompt_chars": [1, MAX_TEXT_CHARS],
    "prompt_tokens": [1, MAX_PROMPT_TOKENS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "context_length": CONTEXT_LENGTH,
    "temperature": "number > 0 (sampling only)",
    "top_p": "number in (0, 1] (sampling only)",
    "seed": "non-negative int, required when do_sample=True",
    "vocab_size": VOCAB_SIZE,
    "eos_token_id": EOS_TOKEN_ID,
    "pad_token_id": PAD_TOKEN_ID,
    "preprocessing": (
        "byte-level BPE with no special tokens; a prompt over MAX_PROMPT_TOKENS, or a prompt whose "
        "tokens plus max_new_tokens exceed CONTEXT_LENGTH, is rejected rather than truncated"
    ),
}


def _check_prompt(prompt: Any) -> str:
    """Raise TypeError/ValueError naming the first violated prompt ceiling; return the prompt."""
    if not isinstance(prompt, str):
        raise TypeError(f"prompt must be a str, got {type(prompt).__name__}")
    if not prompt.strip():
        raise ValueError("prompt must not be empty or whitespace only")
    if len(prompt) > MAX_TEXT_CHARS:
        raise ValueError(f"prompt has {len(prompt)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}")
    return prompt


def validate_inputs(
    prompt: str,
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    do_sample: bool = False,
    temperature: float = 1.0,
    top_p: float = 1.0,
    seed: int | None = None,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Rejection is reported by raising exactly as ``generate`` would: both route the prompt through
    ``_check_prompt`` and the decoding settings through the public ``validate_settings``. The two
    token ceilings (``MAX_PROMPT_TOKENS`` and ``prompt + max_new_tokens <= CONTEXT_LENGTH``) need
    the loaded tokenizer and are therefore enforced inside ``generate``, not here.
    """
    checked = _check_prompt(prompt)
    settings = validate_settings(max_new_tokens, do_sample, temperature, top_p, seed)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry: generate takes one prompt per call")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "prompt-0", "chars": len(checked)}],
        "settings": settings,
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any], references: Sequence[str] | None = None, *, sample_kind: str = "synthetic"
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even though no metric exists here.

    A free-text continuation has no ground truth and the repository ships no metric helper, so the
    verdict is always ``not-measurable`` (EVAL9). ``references`` exists for interface parity with
    the fleet's other pipelines and is recorded in ``reason`` rather than scored: perplexity needs a
    held-out corpus scored by the model, not a reference string compared to one completion, and any
    quality judgement needs human raters or a labelled downstream task.
    """
    settings = result.get("settings", {})
    supplied = references is not None
    return {
        "task": "causal text generation (continuing one prompt)",
        "score_semantics": (
            "the completion carries no score, probability or confidence; `finished_by` says whether "
            "the end-of-text token or the token budget stopped it, and the echoed `settings` say how "
            f"it was decoded ({settings.get('decoding', DECODING_DEFAULT)})"
        ),
        "sample_kind": sample_kind,
        "n_new_tokens": int(result.get("new_tokens", 0)),
        "metrics": [],
        "baselines": [],
        "verdict": "not-measurable",
        "reason": (
            "a continuation has no ground truth and the repository ships no metric helper"
            + (
                "; references were supplied but no metric helper exists to score them here, and a "
                "reference string is not a corpus"
                if supplied
                else "; the evaluated sample has no reference corpus"
            )
        ),
        "needs": (
            "a held-out reference corpus from the deployment domain, scored for perplexity with the "
            "caller's own code, for an intrinsic number; or human raters, or a labelled downstream "
            "task, for any quality or factuality claim — none of which this repository ships"
        ),
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


@dataclass
class GPT2TextGenerationPipeline:
    """``_tokenize`` maps text to token ids, ``_runner`` maps (prompt ids, settings) to new token ids, and
    ``_decode`` maps ids back to text; all three are injectable so tests run offline."""

    _tokenize: Callable[[str], list[int]]
    _runner: Callable[[list[int], dict[str, Any]], list[int]]
    _decode: Callable[[list[int]], str]
    device: str = "cpu"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> GPT2TextGenerationPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs = str(root), {"local_files_only": True}
        elif allow_download:
            source, kwargs = MODEL_ID, {}
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import GPT2LMHeadModel, GPT2TokenizerFast

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        tokenizer = GPT2TokenizerFast.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = GPT2LMHeadModel.from_pretrained(
            source, revision=MODEL_REVISION, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(prompt_ids: list[int], settings: dict[str, Any]) -> list[int]:
            input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=resolved_device)
            gen_kwargs: dict[str, Any] = {
                "max_new_tokens": settings["max_new_tokens"],
                "do_sample": settings["do_sample"],
                "pad_token_id": PAD_TOKEN_ID,
                "eos_token_id": EOS_TOKEN_ID,
            }
            if settings["do_sample"]:
                gen_kwargs.update(temperature=settings["temperature"], top_p=settings["top_p"])
                torch.manual_seed(settings["seed"])
            with torch.inference_mode():
                output = model.generate(input_ids, attention_mask=torch.ones_like(input_ids), **gen_kwargs)
            return output[0, len(prompt_ids) :].tolist()

        def tokenize(text: str) -> list[int]:
            return tokenizer(text, add_special_tokens=False)["input_ids"]

        source_kind = "local-snapshot" if kwargs else "hf-hub"
        return cls(tokenize, runner, tokenizer.decode, resolved_device, source_kind)

    def generate(
        self,
        prompt: str,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        do_sample: bool = False,
        temperature: float = 1.0,
        top_p: float = 1.0,
        seed: int | None = None,
    ) -> dict[str, Any]:
        """Continue one prompt. Greedy (deterministic) unless ``do_sample=True`` with a ``seed``."""
        prompt = _check_prompt(prompt)
        settings = validate_settings(max_new_tokens, do_sample, temperature, top_p, seed)
        prompt_ids = list(self._tokenize(prompt))
        if not 1 <= len(prompt_ids) <= MAX_PROMPT_TOKENS:
            raise ValueError(
                f"prompt has {len(prompt_ids)} tokens, outside 1..MAX_PROMPT_TOKENS={MAX_PROMPT_TOKENS}"
            )
        if len(prompt_ids) + settings["max_new_tokens"] > CONTEXT_LENGTH:
            raise ValueError(
                f"prompt tokens {len(prompt_ids)} + max_new_tokens {settings['max_new_tokens']} "
                f"> CONTEXT_LENGTH={CONTEXT_LENGTH}"
            )
        new_ids = [int(t) for t in self._runner(prompt_ids, settings)]
        if len(new_ids) > settings["max_new_tokens"] or any(not 0 <= t < VOCAB_SIZE for t in new_ids):
            raise RuntimeError("runner returned more than max_new_tokens tokens, or an id outside the vocab")
        finished_by = "eos" if EOS_TOKEN_ID in new_ids else "max_new_tokens"
        kept = new_ids[: new_ids.index(EOS_TOKEN_ID)] if finished_by == "eos" else new_ids
        completion = self._decode(kept) if kept else ""
        return {
            "prompt": prompt,
            "completion": completion,
            "text": prompt + completion,
            "prompt_tokens": len(prompt_ids),
            "new_tokens": len(kept),
            "finished_by": finished_by,
            "settings": settings,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `15`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `607a30d783df…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `GPT2TextGenerationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "gpt2",
  "modelId": "openai-community/gpt2",
  "revision": "607a30d783dfa663caf39e06633721c8d4cfcd7e",
  "files": [
    {
      "path": "README.md",
      "bytes": 8092,
      "sha256": "0fcd631078093c2aa1d93438b898320b8a1167784e2a1ab37b8016e9de8b3c2e"
    },
    {
      "path": "config.json",
      "bytes": 665,
      "sha256": "0daed7749b4f02b8f76240d5444551d7b08712dab4d0adb8239c56ba823bb7b4"
    },
    {
      "path": "generation_config.json",
      "bytes": 124,
      "sha256": "ed0b32ac72c0f5f44a719abb2d7786ea5146c871f83717b7f2018065954de02b"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 548105171,
      "sha256": "248dfc3911869ec493c76e65bf2fcf7f615828b0254c12b473182f0f81d3a707"
    },
    {
      "path": "onnx/config.json",
      "bytes": 879,
      "sha256": "c6d8a78631f7a03a14493eb78d584ba92c8e68c8774b810426b34df1f8a15b10"
    },
    {
      "path": "onnx/generation_config.json",
      "bytes": 119,
      "sha256": "067a873d1d1a67ffa7237a0ad0eebdb547d3f9209793c8aa476ec130815e3c8c"
    },
    {
      "path": "onnx/merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "onnx/special_tokens_map.json",
      "bytes": 99,
      "sha256": "6f50ab5a5a509a1c309d6171f339b196a900dc9c99ad0408ff23bb615fdae7ad"
    },
    {
      "path": "onnx/tokenizer.json",
      "bytes": 2107653,
      "sha256": "cda20b8ca044949aa07ac4078420c80d1a57139d5f9f33700e46fb2d891e7c66"
    },
    {
      "path": "onnx/tokenizer_config.json",
      "bytes": 234,
      "sha256": "551e26ec611d8d0c8edc3ef72e518a38418cb71f40de1347dd486a595e1557d7"
    },
    {
      "path": "onnx/vocab.json",
      "bytes": 798156,
      "sha256": "3ba3c3109ff33976c4bd966589c11ee14fcaa1f4c9e5e154c2ed7f99d80709e7"
    },
    {
      "path": "tokenizer.json",
      "bytes": 1355256,
      "sha256": "8414cab924d8b9b33013f0d221c5862f365ee9be39c5c2bfae8a5a9e970478a6"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 26,
      "sha256": "5e04eb606e3a1583530a42e36c2a6b6615c86f34fe77e44d9ddeb43ff940931f"
    },
    {
      "path": "vocab.json",
      "bytes": 1042301,
      "sha256": "196139668be63f3b5d6574427317ae82f612a97c5d1cdaf36ed2256dbf636783"
    }
  ],
  "totalBytes": 554331411
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = GPT2TextGenerationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Author the synthetic prompt or optional BYOD

The default sample is **synthetic**: one short English prompt written in this cell — the same prompt the model card's CPU smoke used — so it needs no download and contains no personal data. It ships **no reference continuation**, so whatever the model produces is smoke/sanity evidence that the code path works, never a quality measurement and never benchmark evidence. The decoding settings are Colab form parameters: `GREEDY_MAX_NEW_TOKENS` for the deterministic default, and `SAMPLE_MAX_NEW_TOKENS`, `TEMPERATURE`, `TOP_P`, `SEED` for the sampling demonstration; they are validated against the carried module in Section 5 and echoed back by the pipeline in Sections 6 and 7.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file whose whole content (leading and trailing whitespace stripped) is the prompt — at most `MAX_TEXT_CHARS` characters and at most `MAX_PROMPT_TOKENS` BPE tokens, with prompt plus new tokens inside `CONTEXT_LENGTH` (over-long prompts are rejected by the pipeline, not truncated). The upload stays inside this runtime.

In [ ]:
import hashlib
import io

USE_BYOD = False  # @param {type:"boolean"}
GREEDY_MAX_NEW_TOKENS = 32  # @param {type:"integer"}
SAMPLE_MAX_NEW_TOKENS = 16  # @param {type:"integer"}
TEMPERATURE = 0.8  # @param {type:"number"}
TOP_P = 0.9  # @param {type:"number"}
SEED = 7  # @param {type:"integer"}

if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    prompt = io.TextIOWrapper(io.BytesIO(uploaded[sample_name]), encoding='utf-8').read().strip()
    sample_kind = 'BYOD upload'
else:
    prompt = 'The weather in the mountains is usually'
    sample_name = 'synthetic_weather_prompt'
    sample_kind = 'synthetic (authored in this cell; the model card smoke prompt)'
prompt_sha256 = hashlib.sha256(prompt.encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'chars': len(prompt), 'prompt_sha256': prompt_sha256})
print(repr(prompt[:200]))

## 5. Validate the prompt and both decoding settings → input manifest

`validate_inputs` is the pipeline's public validation stage: it routes the prompt through the same private `_check_prompt` that `generate` uses and the decoding settings through the same public `validate_settings`, so a bad `temperature`, `top_p`, `max_new_tokens` or a missing `seed` fails **before** any model work, naming the condition, and a rejection here is a rejection there. It returns an **input manifest** naming the schema and ceilings, the prompt's character count, and the canonical settings the pipeline will echo back — for greedy decoding `temperature`, `top_p` and `seed` are recorded as `None` because they play no role. The manifest is written to `outputs/gpt2_text_generation_input_manifest.json`, and the sampling configuration is validated alongside it so both decoding modes are covered. `CONTEXT_LENGTH` (1024) is the model's positional window and bounds prompt tokens plus new tokens; `MAX_PROMPT_TOKENS` (1023) leaves room for at least one generated token; `MAX_NEW_TOKENS` (256) bounds one call's cost; `MAX_TEXT_CHARS` is the character guard applied before tokenisation; `VOCAB_SIZE` is the output vocabulary; `EOS_TOKEN_ID` and `PAD_TOKEN_ID` are equal by construction — **GPT-2 ships no pad token, so the pipeline fixes `pad_token_id = eos_token_id = 50256` in code** and passes an all-ones attention mask, which is why the single-prompt path raises no padding warning and why a generated `50256` means "end of text", after which the completion is cut. The token count of the prompt can only be checked after tokenisation, so those two ceilings are enforced inside `generate` in Section 6 (the pipeline rejects, never truncates). To show what rejection looks like, the cell also validates an unseeded sampling request and records the pipeline's own error message as a finding.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
ceilings = {'CONTEXT_LENGTH': CONTEXT_LENGTH, 'MAX_PROMPT_TOKENS': MAX_PROMPT_TOKENS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'VOCAB_SIZE': VOCAB_SIZE, 'EOS_TOKEN_ID': EOS_TOKEN_ID, 'PAD_TOKEN_ID': PAD_TOKEN_ID}
print(ceilings)
print({'pad_eos_quirk': f'GPT-2 has no pad token; PAD_TOKEN_ID == EOS_TOKEN_ID == {PAD_TOKEN_ID}: {PAD_TOKEN_ID == EOS_TOKEN_ID}'})
input_manifest = validate_inputs(prompt, max_new_tokens=GREEDY_MAX_NEW_TOKENS, names=[sample_name])
greedy_settings = input_manifest['settings']
sampling_settings = validate_settings(SAMPLE_MAX_NEW_TOKENS, True, TEMPERATURE, TOP_P, SEED)
input_manifest['sampling_settings'] = sampling_settings
# Demonstrate the unseeded-sampling refusal; the finding is recorded, not swallowed.
try:
    validate_inputs(prompt, max_new_tokens=SAMPLE_MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P)
except ValueError as exc:
    input_manifest['findings'].append({'input': 'unseeded-sampling-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/gpt2_text_generation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))
print({'token_ceiling': f'prompt tokens <= MAX_PROMPT_TOKENS={MAX_PROMPT_TOKENS} and prompt + new tokens <= CONTEXT_LENGTH={CONTEXT_LENGTH} are checked by the pipeline after tokenisation; it rejects, never truncates'})

## 6. Generate with the greedy default and confirm it is deterministic

`generate(prompt, max_new_tokens=...)` with the default `do_sample=False` takes the argmax at every step. It returns `completion` (the new text only), `text` (prompt plus completion), `prompt_tokens`, `new_tokens`, `finished_by` (`'eos'` when the model produced the end-of-text token 50256 — the completion is cut there — or `'max_new_tokens'` when the budget ran out), the echoed `settings` (`decoding: 'greedy'`, with `temperature`/`top_p`/`seed` as `None`), the device and the model identity. **Greedy decoding is deterministic on a fixed device and dtype:** the cell calls `generate` twice with the same settings and checks the two completions are byte-identical — a falsifiable check of the reproducibility contract, which does not extend across devices, PyTorch builds or dtypes. Greedy output is also the mode that repeats itself and drifts into generic text; it is the reference mode, not the recommended one for actual use. The model card's smoke observation on this prompt (32 greedy tokens, `finished_by = 'max_new_tokens'`, completion beginning `" good, but the snow is not."`) is one measurement on that host, not an expected value — near-tied logits can flip a token between CPU and CUDA kernels and change everything after it.

In [ ]:
import time

started = time.perf_counter()
greedy = pipe.generate(prompt, max_new_tokens=GREEDY_MAX_NEW_TOKENS)
greedy_elapsed = time.perf_counter() - started
greedy_repeat = pipe.generate(prompt, max_new_tokens=GREEDY_MAX_NEW_TOKENS)
greedy_checks = {
    'settings_echoed_as_validated': greedy['settings'] == greedy_settings,
    'decoding_is_greedy': greedy['settings']['decoding'] == 'greedy',
    'new_tokens_within_budget': 0 <= greedy['new_tokens'] <= GREEDY_MAX_NEW_TOKENS,
    'prompt_tokens_within_ceiling': 1 <= greedy['prompt_tokens'] <= MAX_PROMPT_TOKENS,
    'text_is_prompt_plus_completion': greedy['text'] == greedy['prompt'] + greedy['completion'],
    'finished_by_is_known': greedy['finished_by'] in ('eos', 'max_new_tokens'),
    'greedy_repeat_is_identical': greedy_repeat['completion'] == greedy['completion'],
}
if not all(greedy_checks.values()):
    raise RuntimeError(f'greedy generate output failed a sanity check: {greedy_checks}')
print({key: value for key, value in greedy.items() if key not in ('prompt', 'completion', 'text')})
print({'seconds_first_call': round(greedy_elapsed, 3), 'checks': greedy_checks})
print(f'prompt:     {prompt!r}')
print(f"completion: {greedy['completion']!r}")

## 7. Generate with explicit, seeded nucleus sampling

Sampling is opt-in: `do_sample=True` with `temperature` (rescales the logits; below 1 sharpens, above 1 flattens), `top_p` (nucleus truncation: only the smallest set of tokens whose cumulative probability reaches `top_p` is sampled from) and a **mandatory `seed`** — `validate_settings` refuses unseeded sampling so a sampled result is always reproducible for the same seed on the same host. The pipeline seeds PyTorch's generator immediately before the model call. Every setting is echoed in `settings` (`decoding: 'nucleus-sampling'`) so an exported result records exactly how it was produced (INF9). This cell samples twice with the same seed and checks the completions are identical, then contrasts the sampled completion with the greedy one from Section 6: a different completion is expected (the model card observed `" good, but you need to be careful with your gear and don't go to"` for seed 7 on its host — an observation, not an expected value, because the sampled path depends on the exact floating-point logits of that host). Sampling is the mode usually preferred for actual use because it avoids greedy repetition; it is not "better" in any measured sense here, and a different seed gives a different continuation.

In [ ]:
started = time.perf_counter()
sampled = pipe.generate(prompt, max_new_tokens=SAMPLE_MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, seed=SEED)
sampled_elapsed = time.perf_counter() - started
sampled_repeat = pipe.generate(prompt, max_new_tokens=SAMPLE_MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P, seed=SEED)
sampled_checks = {
    'settings_echoed_as_validated': sampled['settings'] == sampling_settings,
    'decoding_is_nucleus_sampling': sampled['settings']['decoding'] == 'nucleus-sampling',
    'seed_echoed': sampled['settings']['seed'] == SEED,
    'new_tokens_within_budget': 0 <= sampled['new_tokens'] <= SAMPLE_MAX_NEW_TOKENS,
    'text_is_prompt_plus_completion': sampled['text'] == sampled['prompt'] + sampled['completion'],
    'same_seed_reproduces': sampled_repeat['completion'] == sampled['completion'],
}
if not all(sampled_checks.values()):
    raise RuntimeError(f'sampled generate output failed a sanity check: {sampled_checks}')
print({key: value for key, value in sampled.items() if key not in ('prompt', 'completion', 'text')})
print({'seconds_first_call': round(sampled_elapsed, 3), 'checks': sampled_checks})
print(f"greedy:  {greedy['completion'][:120]!r}")
print(f"sampled: {sampled['completion'][:120]!r}")
print({'sampled_differs_from_greedy': sampled['completion'] != greedy['completion'][: len(sampled['completion'])], 'note': 'expected but not asserted; both are unscored continuations'})

## 8. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report — even, as here, when nothing is measurable. The repository ships **no metric helper and reports no performance measure**: a continuation has no ground truth, so the verdict is always `not-measurable` and the report states what would make the task measurable — a held-out reference corpus from the deployment domain scored for perplexity with the caller's own code for an intrinsic number, or human raters or a labelled downstream task for any quality or factuality claim. Supplying a reference string does not change the verdict, because a reference string is not a corpus and no metric helper exists to score it; the helper records that in `reason` rather than inventing a number. The report is written for the greedy result and carries the decoding mode in `score_semantics`, so a reader can see which configuration produced the unscored text. It lands at `outputs/gpt2_text_generation_evaluation_report.json`.

In [ ]:
report = evaluation_report(greedy, sample_kind=sample_kind)
with open('outputs/gpt2_text_generation_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2))
if report['verdict'] == 'not-measurable':
    print('No metric is reported: a continuation has no ground truth, the sample has no reference corpus, and the repository ships no metric helper; perplexity needs a held-out corpus you supply.')

## 9. Export outputs and provenance

Two further files are written under `outputs/` beside the input manifest and the evaluation report. The two completions go to CSV (`outputs/gpt2_text_generation_completions.csv`) with explicit `mode`, `decoding`, `prompt_tokens`, `new_tokens`, `finished_by`, `seed` and `completion` columns, so each completion stays attached to the configuration that produced it. One JSON record (`outputs/gpt2_text_generation_result.json`) preserves the prompt, the greedy result and the sampled result (each with completion, token counts, `finished_by`, and the echoed settings), the determinism and sanity checks, the ceilings in force including the pad/EOS ids, the input manifest, the evaluation report, the sample identity and digest, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, the verified snapshot summary, and the runtime identity (Python, `torch`, `transformers`, device, dtype). No credentials are involved in any step, so none can reach the export.

In [ ]:
import csv

with open('outputs/gpt2_text_generation_completions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['mode', 'decoding', 'prompt_tokens', 'new_tokens', 'finished_by', 'seed', 'completion'])
    for mode, item in (('greedy', greedy), ('sampled', sampled)):
        writer.writerow([mode, item['settings']['decoding'], item['prompt_tokens'], item['new_tokens'], item['finished_by'], item['settings']['seed'], item['completion']])
payload = {
    'prompt': prompt,
    'greedy': {key: greedy[key] for key in ('completion', 'text', 'prompt_tokens', 'new_tokens', 'finished_by', 'settings')},
    'sampled': {key: sampled[key] for key in ('completion', 'text', 'prompt_tokens', 'new_tokens', 'finished_by', 'settings')},
    'sanity_checks': {'greedy': greedy_checks, 'sampled': sampled_checks},
    'seconds': {'greedy_first_call': round(greedy_elapsed, 3), 'sampled_first_call': round(sampled_elapsed, 3)},
    'ceilings': ceilings,
    'completions_file': 'outputs/gpt2_text_generation_completions.csv',
    'input_manifest': input_manifest,
    'evaluation_report': report,
    'sample': {'name': sample_name, 'kind': sample_kind, 'prompt_sha256': prompt_sha256},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': snapshot['path'], 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'dtype': 'float32',
    },
}
with open('outputs/gpt2_text_generation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

Both completions are unscored continuations from a 2019 base language model: they can be false, repetitive, biased or offensive, they carry no confidence or probability, and nothing in the pipeline filters them. Greedy decoding is the deterministic reference mode (identical on repeat, on the same device and dtype); seeded nucleus sampling is the mode usually preferred for use (identical on repeat for the same seed on the same host, different for a different seed or host). Neither is "better" in any measured sense here: the evaluation report is `not-measurable` because none can be computed without a reference corpus (perplexity) or human judgements, and the model card's smoke completions are observations from one host, not expected values. Prompts are rejected above `MAX_PROMPT_TOKENS` or when prompt plus new tokens would exceed the 1024-token window, never truncated; `finished_by = 'eos'` means the model emitted token 50256, which doubles as the pad id because GPT-2 ships no pad token; the pipeline exposes one prompt per call, no chat format, no batching, no logits and no fine-tuning.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model snapshot, validate the demonstrated prompt and both decoding settings against the enforced ceilings, execute the public pipeline path in both decoding modes with the stated determinism properties, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, text quality on any domain, factual reliability, safety for high-consequence use, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3: a staged file is incomplete or altered — delete it from `weights/gpt2/` and rerun Section 3. A `ValueError`/`TypeError` from `validate_inputs` or `validate_settings` in Section 5: a form parameter is out of range (`max_new_tokens` 1..256, `temperature` > 0, `top_p` in (0, 1], integer `seed` >= 0) — fix it and rerun from Section 4. A `ValueError` naming `MAX_PROMPT_TOKENS` or `CONTEXT_LENGTH` in Section 6: the BYOD prompt tokenises too long for the requested budget — shorten it or lower `GREEDY_MAX_NEW_TOKENS`. A `greedy_repeat_is_identical` failure would indicate non-deterministic kernels on the host and should be reported with the runtime identity.

**Next experiments.** Change `SEED` and rerun Section 7 to see a different sampled continuation; set `TEMPERATURE` to 0.3 and 1.5 and compare how conservative or erratic the samples become; raise `GREEDY_MAX_NEW_TOKENS` to 128 and watch greedy decoding repeat itself; upload a paragraph via `USE_BYOD` and inspect `prompt_tokens`; compute perplexity on a small held-out text of your own with your own code as the first step towards the intrinsic number the evaluation report asks for. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: https://github.com/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/gpt2-text-generation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/openai-community/gpt2
- Upstream code: https://github.com/openai/gpt-2
- Language Models are Unsupervised Multitask Learners (Radford et al., 2019; OpenAI technical report, no arXiv identifier): https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf